In [1]:
import pandas as pd
import numpy as np
import pyranges as pr
from scipy.stats import spearmanr
import os
from sklearn.metrics import roc_auc_score, average_precision_score
import matplotlib.pyplot as plt

In [2]:
phyloP = pd.read_csv('../../../results/SV_effect/outputs/Ath_Simulated_DEL_Len_1-50_withPhyloP.tsv', sep='\t')
phyloP = phyloP.drop('mean_pcv2_large', axis = 1)
phyloP.shape

(39976, 5)

In [3]:
def load_and_process_pcv2(size, shift_values=[0, 5, 10, 15, 20]):
   """Load pcv2 files and calculate mean scores"""
   dfs = []
   for shift in shift_values:
       filename = f'../../../results/SV_effect/outputs/Ath_Simulated_DEL_Len_1-50_pcv2-{size}-2nd_shift_{shift}.tsv'
       df = pd.read_csv(filename, sep='\t')
       df['mean'] = df.iloc[:, 11:21].mean(axis=1)
       dfs.append(df['mean'])
   
   return sum(dfs) / len(dfs)

# Process all sizes
for size in ['large', 'medium', 'small']:
    print(f'Processing {size} model......')
    phyloP[f'pcv2_{size}'] = load_and_process_pcv2(size, shift_values=[0, 5, 10, 15, 20])

Processing large model......
Processing medium model......
Processing small model......


In [4]:
models = ['pcv2_large', 'pcv2_medium', 'pcv2_small', 'evo2', 'gpn', 'pcv1']

results = {}
relevant_mask = (phyloP['meanPhyloP'] > 1) | (phyloP['meanPhyloP'] < 0)
filtered_true_labels = np.where(phyloP['meanPhyloP'][relevant_mask] > 1, 1, 0)

In [5]:
filtered_true_labels.sum()

7662

In [6]:
len(filtered_true_labels) - filtered_true_labels.sum()

10413